# 3. Transfer Learning en Colab: El muro de la compatibilidad

En esta práctica vamos a utilizar MediaPipe Model Maker para realizar *Transfer Learning*. Esta herramienta requiere un entorno de Python específico y compatibilidad con ciertas versiones de TensorFlow. Por ello, usaremos Google Colab ejecutando un motor en Python 3.10.

Asegúrate de haber subido el archivo `dataset_gestos.zip` en el panel lateral de Colab (menú Archivos).

## 1. Descomprimir el Dataset
Al subir el archivo `dataset_gestos.zip` (que creaste comprimiendo las carpetas localmente), descomprímelo con este comando para prepararlo para el entrenamiento.

In [ ]:
!unzip -q dataset.zip -d dataset/
print("✅ Dataset descomprimido en la carpeta 'dataset/'.")

replace dataset/dataset/corte_cuchillo/muestra_0.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
✅ Dataset descomprimido en la carpeta 'dataset/'.


## 2. Preparar el entorno Cloud (Resolución de compatibilidad)
MediaPipe Model Maker necesita un entorno con Python 3.10 para funcionar adecuadamente e instalar todos sus módulos de TF sin conflicto. Ejecuta la siguiente celda como tu "Caballo de Troya" y espera un minuto.

In [ ]:
# Bloque de Compatibilidad: Instalación del motor Python 3.10 y MediaPipe
!sudo apt-get update -y > /dev/null
!sudo apt-get install python3.10 python3.10-distutils -y > /dev/null
!wget -q https://bootstrap.pypa.io/get-pip.py
!python3.10 get-pip.py > /dev/null
!python3.10 -m pip install -q mediapipe-model-maker

print("✅ Entorno de Python 3.10 configurado y listo.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ Entorno de Python 3.10 configurado y listo.


## 3. Crear el script de entrenamiento
Se ha creado un script en tu ordenador local llamado `entrenar_gestos.py` con las instrucciones de la práctica. Sube ese archivo al entorno de Colab. No hay necesidad de copiar el código aquí, simplemente sube el archivo .py que acabo de crear.

### Qué hace este script:
A. **Carga y Organización:** Renombra automáticamente tu carpeta "negativa" o "ninguno" a `None` (requisito de MediaPipe) y carga el dataset usando `gesture_recognizer.Dataset.from_folder()`.
B. **Split:** Divide en 80% entrenamiento, 10% validación y 10% test.
C. **HParams:** Configura `epochs=10` y `export_dir`.
D. **Ejecución y Exportación:** Entrena el modelo y lo guarda en la carpeta `modelo_exportado`.

In [ ]:
%%writefile entrenar_gestos.py
import os
from mediapipe_model_maker import gesture_recognizer

# 1. Renombrar la carpeta negativa a 'None' (requisito estricto del modelo)
if os.path.exists('dataset/negativas'):
    os.rename('dataset/negativas', 'dataset/None')
    print("Carpeta 'negativas' renombrada a 'None'.")
elif os.path.exists('dataset/ninguno'):
    os.rename('dataset/ninguno', 'dataset/None')
    print("Carpeta 'ninguno' renombrada a 'None'.")

# 2. Carga y Organización de Datos
dataset_path = "dataset"
print(f"Cargando dataset desde: {dataset_path}")
data = gesture_recognizer.Dataset.from_folder(
    dirname=dataset_path,
    hparams=gesture_recognizer.HandDataPreprocessingParams()
)

# 3. División del Conocimiento (Split)
train_data, rest_data = data.split(0.8)
validation_data, test_data = rest_data.split(0.5)

print(f"Tamaño conjunto de entrenamiento: {len(train_data)}")
print(f"Tamaño conjunto de validación: {len(validation_data)}")
print(f"Tamaño conjunto de test: {len(test_data)}")

# 4. Configuración del Entrenamiento (HParams)
hparams = gesture_recognizer.HParams(
    export_dir="modelo_exportado",
    epochs=10,
    batch_size=2
)

options = gesture_recognizer.GestureRecognizerOptions(hparams=hparams)

# 5. Ejecución (Entrenamiento)
print("\nIniciando entrenamiento...")
model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)

# 6. Evaluación
print("\nEvaluando el modelo...")
loss, acc = model.evaluate(test_data, batch_size=1)
print(f"Test loss:{loss}, Test accuracy:{acc}")

# 7. Exportación
print("\nExportando modelo...")
model.export_model()
print("✅ Modelo 'gesture_recognizer.task' exportado exitosamente en '/modelo_exportado'.")

Overwriting entrenar_gestos.py


## 4. Iniciar el Entrenamiento
Asegúrate de que ya hayas subido tus imágenes y el script al panel de archivos de Colab (y lo hayas descomprimido si usaste `dataset_gestos.zip`).

Luego ejecuta esta celda usando el motor oculto de Python 3.10. Esto puede tardar varios minutos dependiendo del número de imágenes y las épocas.

In [ ]:
!MPLBACKEND=Agg python3.10 entrenar_gestos.py

2026-05-16 14:52:19.485814: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-16 14:52:19.538157: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-16 14:52:19.538272: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-16 14:52:19.539723: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-16 14:52:19.547916: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-16 14:52:19.548263: I tensorflow/core/platform/cpu_feature_guard.cc:1